In [1]:
# 필수 라이브러리 설치
%pip install langchain chromadb langchain-google-genai pypdf PyPDF2 python-dotenv tqdm langchain-community --quiet

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# 필요한 라이브러리 임포트
import os
import warnings
from dotenv import load_dotenv
from tqdm import tqdm

# PDF 처리를 위한 모듈 (원본 코드 기반)
from PyPDF2 import PdfReader
from langchain.document_loaders import PDFPlumberLoader

# LangChain 관련 모듈
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_chroma import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.docstore.document import Document
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser
from langchain.memory import ConversationBufferMemory
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder, HumanMessagePromptTemplate

# FlashRank를 위한 라이브러리 임포트
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import FlashrankRerank

from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from operator import itemgetter

# 경고 메시지 숨기기 (선택 사항)
warnings.filterwarnings("ignore", category=UserWarning)

In [3]:
# --- 0. config ---
# PDF 파일이 있는 폴더 경로 설정
pdf_folder_path = "D:/Dev/FastCampus/AI엔지니어 과정/house_pdf"  # 실제 PDF 파일들이 있는 폴더 경로로 수정하세요.
persist_directory = "chroma_db" # 벡터 DB를 저장할 디렉토리 이름

# Document Loader 설정
READER = PdfReader
LOADER =  PDFPlumberLoader

# Text Splitter 설정
SPLITTER = RecursiveCharacterTextSplitter
CHUNK_SIZE = 500
CHUNK_OVERLAP = 100
LENGTH_FUNCTION = len

# Embedding Model 설정
EMBEDDING_MODEL = "models/embedding-001"

# LLM 설정
LLM_MODEL = "gemini-2.0-flash"
TEMPPERATURE = 0.0

# retriever 설정
FETCH_K = 500
K =20
SEARCH_TYPE = 'mmr' # "similarity", "similarity_score_threshold"


In [4]:
# --- 1. 환경 설정 ---
load_dotenv() # .env 파일 로드

google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    print("오류: .env 파일 또는 환경 변수에서 GOOGLE_API_KEY를 찾을 수 없습니다.")
    exit()
else:
    print("Google API 키 로드 완료.")

# PDF 폴더 존재 확인
if not os.path.isdir(pdf_folder_path):
    print(f"오류: 지정된 PDF 폴더 '{pdf_folder_path}'를 찾을 수 없습니다.")
    exit()

print(f"PDF 폴더 경로: {pdf_folder_path}")
print(f"벡터 DB 저장 경로: {persist_directory}")

Google API 키 로드 완료.
PDF 폴더 경로: D:/Dev/FastCampus/AI엔지니어 과정/house_pdf
벡터 DB 저장 경로: chroma_db


In [5]:
# --- 2. 문서 로드 및 처리 (원본 코드의 함수 사용) ---
def process_single_pdf(filepath, filename):
    """단일 PDF 파일을 처리하고 페이지 목록(Document 객체 리스트)을 반환합니다."""
    # print(f"단일 PDF 파일 처리 시작: {filename}") # tqdm 사용 시 개별 출력은 생략 가능
    pages_with_source = []
    try:
        # 파일 존재 확인
        if not os.path.exists(filepath):
            print(f"경고: 파일이 존재하지 않습니다: {filepath}")
            return []

        # PyPDF2로 메타데이터 추출
        pdf_info = {}
        total_pages = 0
        try:
            with open(filepath, "rb") as f:
                reader = READER(f)
                if reader.is_encrypted:
                    try:
                         # 암호 해제 시도 (비밀번호가 없다면 실패)
                         if reader.decrypt('') == 0: # 0은 실패 의미
                             print(f"경고: 암호화된 PDF 파일(비밀번호 없음) 건너뜁니다: {filename}")
                             return []
                    except Exception as decrypt_error:
                         print(f"경고: 암호화된 PDF 파일 처리 불가 ({decrypt_error}): {filename}")
                         return []

                total_pages = len(reader.pages)

                if reader.metadata:
                    pdf_info['title'] = reader.metadata.get('/Title', '').strip()
                    pdf_info['author'] = reader.metadata.get('/Author', '').strip()
                    # 필요한 다른 메타데이터 필드 추가 가능
                pdf_info = {k: v for k, v in pdf_info.items() if v} # 빈 값 제거
        except Exception as e:
            print(f"경고: PyPDF2 메타데이터 읽기 오류: {filename} - {e}")
            # 메타데이터 로드 실패해도 내용 로드는 시도

        # PDF 내용 로드 (PDFPlumberLoader 사용)
        try:
            loader = LOADER(filepath)
            pages = loader.load() # 페이지별 Document 객체 리스트 반환

            if not pages:
                print(f"경고: {filename}: PDFPlumberLoader가 페이지를 추출하지 못했습니다.")
                return []

            # 메타데이터 병합 및 Document 객체 생성
            for i, page in enumerate(pages):
                merged_metadata = {
                    'source': filepath, # 원본 파일 경로
                    'filename': filename, # 파일명 추가
                    'page': i + 1, # 페이지 번호 (0부터 시작하므로 +1)
                    'total_pages': total_pages
                }
                merged_metadata.update(pdf_info) # PyPDF2로 추출한 정보 추가

                # 원본 페이지의 메타데이터도 존중
                if page.metadata:
                    # source, page 키는 우리가 설정한 것으로 덮어쓰고 나머지는 추가
                    original_meta_filtered = {k: v for k, v in page.metadata.items() if k not in ['source', 'page']}
                    merged_metadata.update(original_meta_filtered)

                pages_with_source.append(
                    Document(
                        page_content=page.page_content,
                        metadata=merged_metadata
                    )
                )
            # print(f"  - {filename}: {len(pages_with_source)} 페이지 Document 생성 (총 {total_pages} 페이지)")
        except Exception as e:
            print(f"경고: PDFPlumberLoader 로딩 오류: {filename} - {e}")
            return []

    except Exception as e:
        print(f"경고: PDF 처리 중 예외 발생: {filename} - {e}")
        return []

    return pages_with_source

In [6]:
# PDF 폴더 내 모든 PDF 파일 처리
def process_pdfs(path):
    """지정된 경로(폴더)에서 PDF 파일을 로드하고 처리합니다."""
    all_pages = []

    if not os.path.isdir(path):
        print(f"경고: 폴더 경로 '{path}'가 유효하지 않습니다.")
        return []

    pdf_files = [f for f in os.listdir(path) if f.lower().endswith('.pdf')]
    if not pdf_files:
        print(f"경고: '{path}' 폴더에 PDF 파일이 없습니다.")
        return []

    print(f"처리할 PDF 파일 {len(pdf_files)}개 발견.")

    for pdf_filename in tqdm(pdf_files, desc="PDF 처리 중"):
        filepath = os.path.join(path, pdf_filename)
        pages = process_single_pdf(filepath, pdf_filename)
        all_pages.extend(pages)

    print(f"총 {len(all_pages)} 페이지 (Document 객체) 처리 완료.")
    return all_pages

# PDF 처리 실행
global_pages = process_pdfs(pdf_folder_path)

if global_pages:
    print("처리된 첫 페이지 메타데이터 예시:", global_pages[0].metadata)

처리할 PDF 파일 16개 발견.


PDF 처리 중: 100%|██████████| 16/16 [01:52<00:00,  7.05s/it]

총 415 페이지 (Document 객체) 처리 완료.
처리된 첫 페이지 메타데이터 예시: {'source': 'D:/Dev/FastCampus/AI엔지니어 과정/house_pdf\\(영등포복합청사)행복주택 입주자 모집 공고문(2024.5.31.).pdf', 'filename': '(영등포복합청사)행복주택 입주자 모집 공고문(2024.5.31.).pdf', 'page': 1, 'total_pages': 19, 'file_path': 'D:/Dev/FastCampus/AI엔지니어 과정/house_pdf\\(영등포복합청사)행복주택 입주자 모집 공고문(2024.5.31.).pdf', 'Creator': 'Hwp 2020 11.0.0.7571', 'Producer': 'Hancom PDF 1.3.0.546', 'CreationDate': "D:20240520135921+09'00'", 'ModDate': "D:20240520135921+09'00'", 'PDFVersion': '1.4'}


In [7]:
# --- 3. 문서 분할 ---
if not global_pages:
     print("오류: 분할할 문서가 없습니다. PDF 처리 단계를 확인하세요.")
     exit()

text_splitter = SPLITTER(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=LENGTH_FUNCTION,
)
split_docs = text_splitter.split_documents(global_pages)

print(f"총 {len(global_pages)} 페이지를 {len(split_docs)}개의 청크로 분할 완료.")
if split_docs:
    print("분할된 첫 청크 메타데이터 예시:", split_docs[0].metadata)

총 415 페이지를 1741개의 청크로 분할 완료.
분할된 첫 청크 메타데이터 예시: {'source': 'D:/Dev/FastCampus/AI엔지니어 과정/house_pdf\\(영등포복합청사)행복주택 입주자 모집 공고문(2024.5.31.).pdf', 'filename': '(영등포복합청사)행복주택 입주자 모집 공고문(2024.5.31.).pdf', 'page': 1, 'total_pages': 19, 'file_path': 'D:/Dev/FastCampus/AI엔지니어 과정/house_pdf\\(영등포복합청사)행복주택 입주자 모집 공고문(2024.5.31.).pdf', 'Creator': 'Hwp 2020 11.0.0.7571', 'Producer': 'Hancom PDF 1.3.0.546', 'CreationDate': "D:20240520135921+09'00'", 'ModDate': "D:20240520135921+09'00'", 'PDFVersion': '1.4'}


In [19]:
# --- 4. 임베딩 및 벡터 저장 ---
try:
    embeddings = GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL, google_api_key=google_api_key)
    print("Gemini 임베딩 모델 초기화 완료.")
except Exception as e:
    print(f"Gemini 임베딩 모델 초기화 오류: {e}")
    exit()

if os.path.exists(persist_directory):
    print(f"기존 벡터 저장소 '{persist_directory}'를 로드합니다.")
    try:
        vectordb = Chroma(persist_directory=persist_directory, embedding_function=embeddings)
        print(f"기존 벡터 저장소 로드 완료. 저장된 문서 수: {vectordb._collection.count()}")
    except Exception as e:
        print(f"기존 벡터 저장소 로드 오류: {e}")
        print("벡터 저장소를 새로 생성합니다.")
        if not split_docs:
            print("오류: 벡터 저장소를 생성할 분할된 문서가 없습니다.")
            exit()
        vectordb = Chroma.from_documents(
            documents=split_docs,
            embedding_function=embeddings,
            persist_directory=persist_directory
        )
        print(f"새로운 벡터 저장소 생성 및 저장 완료. 저장된 문서 수: {vectordb._collection.count()}")
else:
    if not split_docs:
        print("오류: 벡터 저장소를 생성할 분할된 문서가 없습니다.")
        exit()
    print(f"'{persist_directory}'에 새로운 벡터 저장소를 생성하고 문서를 저장합니다.")
    vectordb = Chroma.from_documents(
        documents=split_docs,
        embedding_function=embeddings,
        persist_directory=persist_directory
    )
    print(f"새로운 벡터 저장소 생성 및 저장 완료. 저장된 문서 수: {vectordb._collection.count()}")


Gemini 임베딩 모델 초기화 완료.
기존 벡터 저장소 'chroma_db'를 로드합니다.
기존 벡터 저장소 로드 완료. 저장된 문서 수: 1741


In [9]:
# --- 5. LLM 및 Retriever 초기화 ---
try:
    llm = ChatGoogleGenerativeAI(
        model=LLM_MODEL,
        google_api_key=google_api_key,
        temperature=TEMPPERATURE, # Few-shot 예제를 따르도록 낮은 온도 설정 권장
        convert_system_message_to_human=True
    )
    print("Gemini LLM 초기화 완료.")
except Exception as e:
    print(f"Gemini LLM 초기화 오류: {e}")
    exit()

retriever = vectordb.as_retriever(
    search_type=SEARCH_TYPE,
    search_kwargs={'fetch_k': FETCH_K, 'k': K} # fetch_K : 후보 집합으로 선택되는 문서의 수 / k : 최종적으로 선택되는 문서의 수
)
print(f"Retriever 초기화 완료")

Gemini LLM 초기화 완료.
Retriever 초기화 완료


In [15]:
# --- 5.1 Reranker 초기화 ---

try:
    # FlashRank 압축기 초기화 시 모델 이름 지정 제거
    compressor = FlashrankRerank()  # model_name 인자 제거
    
    # 원래 retriever를 압축 retriever로 감싸기
    compression_retriever = ContextualCompressionRetriever(
        base_compressor=compressor,
        base_retriever=retriever,
    )
    print(f"FlashRank Reranker 초기화 완료.") # 모델 이름 출력 제거
except Exception as e:
    print(f"FlashRank Reranker 초기화 오류: {e}")
    print("원래 Retriever를 계속 사용합니다.")
    compression_retriever = retriever  # 오류 발생 시 원래 retriever를 사용


FlashRank Reranker 초기화 완료.


In [16]:
# --- 메모리 초기화 ---
# 대화 기록을 저장할 메모리 초기화
memory = ConversationBufferMemory(
    memory_key="chat_history",  # 프롬프트 템플릿에서 참조할 키
    return_messages=True,       # 메시지 형식으로 반환
    output_key="answer"         # 저장할 출력 키
)

In [17]:
# 프롬프트 설정
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", """
당신은 한국의 부동산 정책과 입주 공고 전문가입니다. 제공된 "문맥 정보"와 "대화 기록"을 바탕으로 "질문"에 대해 답변해주세요.

지침:
- 답변은 무조건 한국어로만 답하세요.
- 너무 길거나 복잡한 답변은 피하세요. 사용자가 이해하기 쉽도록 작성하세요.
- "문맥 정보"에서 질문에 대한 답을 찾으세요.
- 답을 찾을 수 없는 경우, "죄송합니다, 제공된 문서에서 해당 질문에 대한 답변을 찾을 수 없습니다."라고 명확히 답변하세요.
- 절대 추측하거나 주어진 정보 외의 내용을 답변하지 마세요.
- 답변 생성 시 반드시 제공된 "문맥 정보"만을 근거로 하세요.
- 답변은 친절하고 명확하게 한국어로 작성해주세요.
- 이전 대화 내용을 참고하여 맥락을 유지하세요.
- 가능하다면, 답변의 근거가 된 문서의 출처(파일명, 페이지 번호)를 답변 끝에 간략히 언급하세요. (예: [출처: 파일명.pdf, 페이지번호])
"""),
    MessagesPlaceholder(variable_name="chat_history"),  # 대화 기록 삽입
    ("human", """
문맥 정보:
{context}

질문:
{question}
"""),
])

In [18]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 2. 수정된 LCEL 체인 구성

# 입력 ({"question": str})을 받아 필요한 모든 정보를 준비하는 단계
prepare_chain = RunnableParallel(
    # itemgetter('question')으로 질문 문자열만 추출하여 retriever에 전달, 결과를 source_documents에 저장
    source_documents=itemgetter('question') | compression_retriever,
    # 질문 문자열을 그대로 question 키에 전달
    question=itemgetter('question'),
    # 메모리에서 대화 기록 로드 (입력 dict를 받지만 사용하지 않음)
    chat_history=RunnableLambda(lambda x: memory.load_memory_variables({})["chat_history"]),
).assign( # 검색된 문서를 포맷하여 context 키 추가 (위 결과들을 입력으로 받음)
    context=RunnableLambda(lambda x: format_docs(x['source_documents']))
)
# 이 단계의 출력 예시: {'source_documents': [Doc...], 'question': str, 'chat_history': str, 'context': str}


# 준비된 정보를 받아 최종 답변을 생성하고 출력 형식을 맞추는 단계
rag_chain_with_source = (
    prepare_chain
    | RunnableParallel(
        # 'result' 키 생성: 필요한 context, question, chat_history만 prompt에 전달 후 LLM 호출 및 파싱
        result=(
            # 필요한 키만 선택하여 prompt에 전달 (딕셔너리 형태 유지)
            RunnableLambda(lambda x: {"context": x["context"], "question": x["question"], "chat_history": x["chat_history"]})
            | chat_prompt
            | llm
            | StrOutputParser() # 표준 문자열 출력 파서 사용
        ),
        # 'source_documents' 키는 그대로 전달
        source_documents=itemgetter('source_documents')
    )
).with_config({"run_name": "house_rag_chain_with_flashrank"}) # 설정 추가

print("LCEL 기반 RAG 체인 구성 완료 (수정됨 - 단일 검색 최적화).")

LCEL 기반 RAG 체인 구성 완료 (수정됨 - 단일 검색 최적화).


In [22]:
# --- 7. 질의응답 실행 (LCEL 방식으로 변경) ---
print("주택 공고에 대해 질문해주세요. 종료하려면 '종료'를 입력하세요.")

while True:
    try:
        user_question = input("\n👤 질문: ")
        if user_question.lower() == "종료":
            print("\n🤖 챗봇: 이용해주셔서 감사합니다. 종료합니다.")
            break
        if not user_question.strip():
            print("질문을 입력해주세요.")
            continue

        print("\n⏳ 답변 생성 중...")

        # LCEL 체인 호출
        response = rag_chain_with_source.invoke({
            "question": user_question
        })

        result = response["result"]

        # 사용자 질문 출력 (위치 변경)
        print(f"\n🙋‍♂️ 사용자 질문: {user_question}")

        print("\n🤖 답변:")
        print(result)

        # 메모리에 대화 저장
        memory.save_context(
            {"question": user_question},
            {"answer": result}
        )

        # 소스 문서 처리 (기존 코드, 변경 없음)
        print("\n📚 근거 문서:")
        if "source_documents" in response:
            source_documents = response["source_documents"]
            num_documents = len(source_documents)
            top_n = 3  # 상위 3개 문서만 상세히 출력

            if num_documents > 0:
                print(f"총 {num_documents}개의 관련 문서가 있습니다.")
                if num_documents <= top_n:
                    for i, doc in enumerate(source_documents):
                        print(f"\n--- 문서 {i+1} ---")
                        print(f"- 파일: {doc.metadata.get('filename', '알 수 없음')}, 페이지: {doc.metadata.get('page', '알 수 없음')}")
                        # 필요하다면 doc.page_content를 출력할 수도 있습니다.
                        # print(f"내용: {doc.page_content}")
                else:
                    print(f"\n--- 상위 {top_n}개 문서 ---")
                    for i in range(top_n):
                        doc = source_documents[i]
                        print(f"\n--- 문서 {i+1} ---")
                        print(f"- 파일: {doc.metadata.get('filename', '알 수 없음')}, 페이지: {doc.metadata.get('page', '알 수 없음')}")
                        # 필요하다면 doc.page_content를 출력할 수도 있습니다.
                        # print(f"내용: {doc.page_content}")

                    remaining_count = num_documents - top_n
                    print(f"\n--- 그 외 {remaining_count}개 문서 ---")
                    print(f"유사한 문서가 {remaining_count}개 더 있습니다.")
            else:
                print("근거 문서를 찾을 수 없습니다.")
        else:
            print("근거 문서를 찾을 수 없습니다.")

    except Exception as e:
        print(f"\n⚠️ 처리 중 오류 발생: {e}")
    except KeyboardInterrupt:
        print("\n\n🤖 챗봇: 사용자에 의해 중단되었습니다. 종료합니다.")
        break


주택 공고에 대해 질문해주세요. 종료하려면 '종료'를 입력하세요.

⏳ 답변 생성 중...

🙋‍♂️ 사용자 질문: 안녕, 내 이름은 케빈이고 35세 수서에 사는 남성이야.

🤖 답변:
안녕하세요, 케빈님. 무엇을 도와드릴까요?

📚 근거 문서:
총 3개의 관련 문서가 있습니다.

--- 문서 1 ---
- 파일: {공고문(PDF)}_관악봉천행복주택입주자모집공고문.pdf, 페이지: 28

--- 문서 2 ---
- 파일: LH수서1단지(수서역세권A1BL)행복주택예비입주자모집공고문.pdf, 페이지: 1

--- 문서 3 ---
- 파일: {공고문(PDF)}_관악봉천행복주택입주자모집공고문.pdf, 페이지: 11

⏳ 답변 생성 중...

🙋‍♂️ 사용자 질문: 내 이름과 나이가 뭐라고?

🤖 답변:
케빈님, 35세이십니다.

📚 근거 문서:
총 3개의 관련 문서가 있습니다.

--- 문서 1 ---
- 파일: {공고문(PDF)}_서울공릉 양원S1BL행복주택예비입주자모집공고문(2024.05.29.).pdf, 페이지: 10

--- 문서 2 ---
- 파일: 240517서울가좌·휘경행복주택예비입주자모집공고문.pdf, 페이지: 16

--- 문서 3 ---
- 파일: {공고문(PDF)}_관악봉천행복주택입주자모집공고문.pdf, 페이지: 26

⏳ 답변 생성 중...

🙋‍♂️ 사용자 질문: 나에게 적합한 행복주택 공고가 있어?

🤖 답변:
죄송합니다, 제공된 문서에서 케빈님께 적합한 행복주택 공고 정보를 찾을 수 없습니다. 문서에는 행복주택 공고 자체가 나와있지 않습니다.

📚 근거 문서:
총 3개의 관련 문서가 있습니다.

--- 문서 1 ---
- 파일: LH수서1단지(수서역세권A1BL)행복주택예비입주자모집공고문.pdf, 페이지: 14

--- 문서 2 ---
- 파일: {공고문(PDF)}_(공고문)서울독산13(광명하안13)행복주택입주자격완화모집공고문(241031).pdf, 페이지: 6

--- 문서 3 ---
- 파일: 240517서울가좌·휘경행복주택예비입주자모집공